# CSV: structured similarity groups

Experimental notebook interface to `extension.clustering.pipeline`. The existing implementation remains authoritative. Run cells from top to bottom in a fresh kernel. Outputs are isolated under `output/notebook_extension/`; rerunning replaces that notebook’s outputs.

This is a staged orchestration of existing functions, not an independent reimplementation. Open a function’s source to change its algorithm; substitute a compatible function call below to test a variation. Preserve train-only fitting, splits, matched costs, support thresholds and dependence-aware uncertainty. See the repository methodology before interpreting results.

## 1. Locate the project and import reusable functions
The Python environment must contain the project dependencies plus a Jupyter kernel.

In [ ]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts in this folder or anywhere inside the repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/ds340-notebook-mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/ds340-notebook-cache")
from extension.clustering.pipeline import (  # noqa: E402 - repository path must be set before project imports
    SEED,
    Path,
    argparse,
    embed,
    group_contracts,
    hashlib,
    json,
    load_contracts,
    pd,
    threadpool_limits,
    write_reviews,
)

SOURCE_FILE = ROOT / "extension/clustering/pipeline.py"

## 2. Configuration
Defaults match the existing CLI, except for isolated paths. Edit this argument list for a separate sensitivity run, using a new output directory. Do not select a model using the held-out results.

In [ ]:
NOTEBOOK_ARGUMENTS = ["--data", "data/kalshi_recent_year_csv", "--output", "output/notebook_extension/clustering"]

## Parse and validate settings

In [ ]:
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--data", type=Path, default=Path("data/kalshi_recent_year_csv"))
parser.add_argument("--output", type=Path, default=Path("output/clustering"))
parser.add_argument("--backend", choices=["lsa", "sentence"], default="lsa")
parser.add_argument("--model", type=Path)
parser.add_argument("--method", choices=["hierarchical", "kmeans"], default="hierarchical")
parser.add_argument("--fit-limit", type=int, default=20000)
parser.add_argument("--family-fit-limit", type=int, default=1000)
parser.add_argument("--clusters", type=int, default=8, help="K-means clusters per family")
parser.add_argument("--distance", type=float, default=0.35, help="Average-linkage cosine distance cutoff")
parser.add_argument("--min-similarity", type=float, default=0.75)
parser.add_argument("--min-contracts", type=int, default=20)
parser.add_argument("--min-events", type=int, default=10)
args = parser.parse_args(NOTEBOOK_ARGUMENTS)
if min(args.fit_limit, args.family_fit_limit, args.clusters, args.min_contracts, args.min_events) < 1:
    parser.error("Counts and limits must be positive")
if not 0 < args.distance <= 2 or not 0 <= args.min_similarity <= 1:
    parser.error("Require distance in (0, 2] and minimum similarity in [0, 1]")

# Keep experimental writes away from the established implementation and results.
prototype_root = (ROOT / "output/notebook_extension").resolve()
if prototype_root not in args.output.resolve().parents:
    raise ValueError("Choose an output subdirectory under output/notebook_extension/.")
args.output.mkdir(parents=True, exist_ok=True)
(args.output / "run_manifest.json").write_text(
    json.dumps({"completed": False, "interface": "experimental_notebook"}, indent=2)
)

## Create the isolated run directory

In [ ]:
args.output.mkdir(parents=True, exist_ok=True)

## Record input provenance

In [ ]:
manifest = {
    "completed": False,
    "config": {k: str(v) if isinstance(v, Path) else v for (k, v) in vars(args).items()},
    "seed": SEED,
    "input_files": [],
}
manifest["completed"] = False
manifest["notebook_source_sha256"] = hashlib.sha256(
    (ROOT / "notebooks/extension_prototype/02_csv_clustering.ipynb").read_bytes()
).hexdigest()
manifest["interface"] = "experimental_notebook"
for split in ["train", "validation", "test"]:
    path = args.data / "split" / split / "markets.csv"
    manifest["input_files"].append(
        {"path": str(path), "bytes": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
    )
manifest_path = args.output / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

## Load eligible metadata and audit splits

In [ ]:
with threadpool_limits(limits=2):
    (frame, manifest["attrition"]) = load_contracts(args.data)
    print(f"Representing {len(frame):,} contracts using {args.backend}", flush=True)

## Fit the training encoder and transform held-out text

In [ ]:
with threadpool_limits(limits=2):
    vectors = embed(frame, args)

## Assign frozen family clusters and export review tables

In [ ]:
with threadpool_limits(limits=2):
    result = group_contracts(frame, vectors, args)
    result.to_csv(args.output / "assignments.csv", index=False)
    write_reviews(result, args.output)

## Finalize the run

In [ ]:
manifest["completed"] = True
manifest["note"] = "Heuristic families; event-disjoint supplied split, not chronological or unseen-series evaluation."
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(result.groupby(["split", "status"]).size().to_string())
print(f"Results: {args.output}")

## Inspect results and limitations
A completed manifest means execution finished; it does not validate the research assumptions. This is not a chronological backtest, a causal wording experiment, or an execution-aware trading simulation. Sparse/rejected groups must remain visible. Compare this run with the CLI using identical inputs, settings and software before adopting the notebook route.

In [ ]:
from IPython.display import Image, display

result_dir = args.output.resolve()
print("Results:", result_dir)
print("Completed:", json.loads((result_dir / "run_manifest.json").read_text()).get("completed", False))
for csv_name in ["model_comparison.csv", "coverage.csv", "population_coverage.csv", "heterogeneity_tests.csv"]:
    csv_path = result_dir / csv_name
    if csv_path.exists():
        print(csv_name)
        display(pd.read_csv(csv_path).head(20))
for image_path in sorted(result_dir.glob("*.png")) + sorted((result_dir / "figures").glob("*.png")):
    display(Image(filename=str(image_path)))
report_path = result_dir / "FINDINGS.md"
if not report_path.exists():
    report_path = result_dir / "STRUCTURAL_FINDINGS.md"
if report_path.exists():
    print("Report:", report_path)